# J4: Pillar ablations
Run Pillar 1 first (load->price ladder): off / external Module A quantiles / internal pathway / true-load oracle. Then Pillar 2 (spike head + asinh + tail weighting), aux heads, loss weighting (dwa/uncertainty/fixed), RevIN. Deferred: JSU head, PLE/MMoE rung, PCGrad (see spec).

In [ ]:
import sys; sys.path.insert(0, '../../src')
import os; os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'
import pandas as pd, numpy as np, torch
from data.loaders import load_split
from module_joint.config import JointConfig, select_device
LQ = pd.read_parquet('../../data/module_a/load_quantiles.parquet')
device = select_device(); print('device', device)


In [ ]:
from module_joint.train import train_one
from module_joint.evaluate import compare_price, make_truth, leaderboard
train, val = load_split('train'), load_split('val')
val_ctx = pd.concat([train.tail(168), val])
ladders = {
  'no_load':   dict(use_load_quantiles=False, load_to_price=False),
  'ext_loadq': dict(use_load_quantiles=True,  load_to_price=False),
  'internal':  dict(use_load_quantiles=True,  load_to_price=True),
}
rows = []
for name, flags in ladders.items():
    cfg = JointConfig(backbone='tide', max_epochs=60, **flags)
    est,_ = train_one(cfg, train, val, LQ, device=device)
    pr = est.predict_quantiles(val_ctx, LQ, restrict_to=val.index)['price']
    tr = make_truth(val_ctx, pr.index, 'price')
    m = compare_price(pr, tr)['overall']; m['variant']=name; rows.append(m)
leaderboard(rows)
